# Exercice 2 : Vérifier l'intégrité d'un modèle, puis changer de format

> **Objectif :** savoir ce que contient le fichier d'un modèle du Ch. 1, détecter qu'il a été modifié (empreinte SHA-256, signature),
> puis le convertir dans un format qui ne contient **que des données** (ONNX ici, safetensors pour les réseaux de neurones).

**Le scénario.** Notre détecteur de maliciels du Ch. 1 est publié dans un registre de modèles interne. Un attaquant obtient un accès en écriture à ce registre (compte volé, CI/CD compromise comme pour Ultralytics en 2024).
Il ne vole rien : il **modifie** discrètement le modèle.

**Le plan.**
1. Entraîner et sauvegarder les trois modèles du Ch. 1 (arbre, forêt, KNN).
2. Regarder ce que contient chaque fichier, donc ce que l'attaquant peut modifier.
3. Calculer les empreintes, puis laisser l'attaquant modifier le KNN.
4. Détecter la modification (empreinte), puis résister à un attaquant qui recalcule l'empreinte (signature).
5. Convertir en ONNX et comparer ; regarder safetensors sur un vrai modèle de Hugging Face.

## Étape 0 : exécutez cette cellule en premier
Elle liste les dépendances. Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances : scikit-learn pandas skl2onnx onnx onnxruntime cryptography huggingface_hub
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 6 - Chaîne d'approvisionnement des modèles"
    !pip -q install skl2onnx onnxruntime
print('Prêt :', os.path.exists('helper.py'))

Prêt : True


## Étape 1 : Entraîner et sauvegarder les modèles du Ch. 1
**Ce qu'on fait :** on reprend le jeu de données du Ch. 1 (19 611 exécutables Windows, 77 champs de l'en-tête PE) et les trois modèles vus en classe.
On les sauvegarde au format pickle, comme 90 % des projets scikit-learn.

In [2]:
import os, pickle, pandas as pd, numpy as np
from helper import charger_maliciels, entrainer_modeles, MODELES

X_train, X_test, y_train, y_test = charger_maliciels()
modeles = entrainer_modeles(X_train, y_train)

os.makedirs(MODELES, exist_ok=True)
for nom, m in modeles.items():
    with open(f'{MODELES}/{nom}.pkl', 'wb') as f:
        pickle.dump(m, f)
    print(f"{nom:<6} accuracy = {m.score(X_test, y_test):.4f} | fichier = {os.path.getsize(f'{MODELES}/{nom}.pkl')/1e6:.2f} Mo")

arbre  accuracy = 0.9875 | fichier = 0.01 Mo
foret  accuracy = 0.9898 | fichier = 1.41 Mo


knn    accuracy = 0.9845 | fichier = 4.96 Mo


## Étape 2 : Que contient le fichier d'un modèle ?
**Ce qu'on fait :** on ouvre chaque modèle (ce sont les nôtres, donc de confiance) et on regarde ce qui a été sauvegardé.

In [3]:
arbre, foret, knn = modeles['arbre'], modeles['foret'], modeles['knn']
t = arbre.tree_
print(f"ARBRE : {t.node_count} nœuds. Chaque nœud = (feature, seuil, enfant gauche, enfant droit, votes).")
print(f"   Nœud racine : si {X_train.columns[t.feature[0]]} <= {t.threshold[0]} alors à gauche, sinon à droite.")
print(f"FORÊT : {len(foret.estimators_)} arbres, {sum(e.tree_.node_count for e in foret.estimators_)} nœuds au total.")
print(f"KNN   : _fit_X de forme {knn._fit_X.shape} et _y de forme {knn._y.shape}.")
print(f"   Données d'entraînement : {X_train.shape}. Le KNN a sauvegardé TOUTE la base d'entraînement.")

ARBRE : 107 nœuds. Chaque nœud = (feature, seuil, enfant gauche, enfant droit, votes).
   Nœud racine : si MajorSubsystemVersion <= 5.5 alors à gauche, sinon à droite.
FORÊT : 50 arbres, 17402 nœuds au total.
KNN   : _fit_X de forme (15688, 77) et _y de forme (15688,).
   Données d'entraînement : (15688, 77). Le KNN a sauvegardé TOUTE la base d'entraînement.


**👀 Ce qu'on doit voir :**
- l'arbre = une centaine de **seuils** ; changer un seul seuil change les décisions de toute une branche ;
- la forêt = des milliers de seuils ;
- le KNN = **la base d'entraînement complète** (15 688 exécutables). Le fichier est donc aussi une fuite de données, et y ajouter quelques lignes suffit à changer ses décisions.

**❓ Question pour la classe :** lequel des trois est le plus facile à piéger sans que la précision bouge ? *Réponse : le KNN. On ajoute des points, on ne touche à rien d'autre.*

## Étape 3 : Publier les empreintes, puis l'attaque
**Ce qu'on fait :** le défenseur publie un **manifeste** : le SHA-256 de chaque fichier. Puis l'attaquant, qui a accès au registre, prend **son** maliciel (bien détecté),
ajoute 5 copies étiquetées « bénin » dans la base du KNN, et réécrit `knn.pkl`.

In [4]:
from helper import ecrire_manifeste, verifier_manifeste
fichiers = [f'{MODELES}/{n}.pkl' for n in modeles]
ecrire_manifeste(fichiers, f'{MODELES}/manifeste.json')

{'arbre.pkl': '5e405dd4ef30d738c2f2ab854aa0a4516a60e69405f2d31760aaa7050b3080b8',
 'foret.pkl': '3376c5cbf89ec8c66b9de394006beb8c0cfa7325f5c2dad7de482e5dfa7e40db',
 'knn.pkl': '6fe76f00573e754171b02cc7caa017c36dadb3ab66c601cc0a8fb9146bca06f3'}

In [5]:
import copy
mon_maliciel = X_test[(y_test == 1) & (knn.predict(X_test) == 1)].iloc[[0]]     # un maliciel bien détecté

knn_piege = copy.deepcopy(knn)
knn_piege._fit_X = np.vstack([knn_piege._fit_X, np.repeat(mon_maliciel.values, 5, axis=0)])
knn_piege._y = np.concatenate([knn_piege._y, [0] * 5])                            # 0 = « bénin »
with open(f'{MODELES}/knn.pkl', 'wb') as f:                                        # l'attaquant écrase le fichier publié
    pickle.dump(knn_piege, f)

print(f"Accuracy avant : {knn.score(X_test, y_test):.4f} | après : {knn_piege.score(X_test, y_test):.4f}")
print(f"Le maliciel de l'attaquant : avant = {knn.predict(mon_maliciel)[0]} | après = {knn_piege.predict(mon_maliciel)[0]}  (1 = maliciel, 0 = bénin)")

Accuracy avant : 0.9845 | après : 0.9842
Le maliciel de l'attaquant : avant = 1 | après = 0  (1 = maliciel, 0 = bénin)


**👀 Ce qu'on doit voir :** l'accuracy ne bouge presque pas (0,9845 → 0,9842), mais le maliciel de l'attaquant passe de 1 à **0** (« bénin »).
Aucun test de performance ne verra cette modification.

## Étape 4 : Détecter la modification
**Ce qu'on fait :** avant de déployer, on recalcule les empreintes et on les compare au manifeste.

In [6]:
verifier_manifeste(f'{MODELES}/manifeste.json', MODELES)

,fichier,attendu,obtenu,intègre
0,arbre.pkl,5e405dd4ef30d738…,5e405dd4ef30d738…,True
1,foret.pkl,3376c5cbf89ec8c6…,3376c5cbf89ec8c6…,True
2,knn.pkl,6fe76f00573e7541…,50bf97b815900a2f…,False


**👀 Ce qu'on doit voir :** `knn.pkl` est **non intègre** (`False`). Les deux autres sont intacts. Un seul octet modifié change complètement le SHA-256.

**Mais** : si l'attaquant peut écrire `knn.pkl`, il peut aussi réécrire `manifeste.json` avec la nouvelle empreinte. L'empreinte prouve l'**intégrité**, pas l'**origine**.
Il faut une **signature** : seule la personne qui détient la clé privée peut produire un manifeste valide.

In [7]:
import json
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature

cle_privee = Ed25519PrivateKey.generate()            # gardée par l'équipe ML (idéalement dans un HSM / KMS)
cle_publique = cle_privee.public_key()               # distribuée à ceux qui déploient

# Le défenseur republie le bon modèle et signe le manifeste.
with open(f'{MODELES}/knn.pkl', 'wb') as f:
    pickle.dump(knn, f)
manifeste = json.dumps(ecrire_manifeste(fichiers, f'{MODELES}/manifeste.json'), sort_keys=True).encode()
signature = cle_privee.sign(manifeste)

# L'attaquant remplace le modèle ET recalcule le manifeste.
with open(f'{MODELES}/knn.pkl', 'wb') as f:
    pickle.dump(knn_piege, f)
faux_manifeste = json.dumps(ecrire_manifeste(fichiers, f'{MODELES}/manifeste.json'), sort_keys=True).encode()

for nom, contenu in [('manifeste original', manifeste), ("manifeste de l'attaquant", faux_manifeste)]:
    try:
        cle_publique.verify(signature, contenu)
        print(f'{nom:<26} : signature VALIDE')
    except InvalidSignature:
        print(f'{nom:<26} : signature INVALIDE -> on refuse de déployer')

manifeste original         : signature VALIDE
manifeste de l'attaquant   : signature INVALIDE -> on refuse de déployer


In [8]:
# On remet le bon modèle en place pour la suite.
with open(f'{MODELES}/knn.pkl', 'wb') as f:
    pickle.dump(knn, f)
ecrire_manifeste(fichiers, f'{MODELES}/manifeste.json');

**👀 Ce qu'on doit voir :** le manifeste de l'attaquant est cohérent avec son fichier, mais sa signature est **invalide** : il n'a pas la clé privée.
C'est le principe de l'outil `model-signing` (OpenSSF, v1.0 en avril 2025), qui ajoute l'identité du signataire (Sigstore) et un journal public.

## Étape 5 : Changer de format : ONNX
**Ce qu'on fait :** un pickle contient des instructions Python. ONNX contient un **graphe d'opérateurs mathématiques** standard et des tableaux de nombres.
On convertit les trois modèles avec `skl2onnx`, puis on compare.

In [9]:
import onnx, onnxruntime as ort
from skl2onnx import to_onnx

lignes = []
for nom, m in modeles.items():
    onx = to_onnx(m, X_train.values[:1], options={id(m): {'zipmap': False}})
    onnx.save(onx, f'{MODELES}/{nom}.onnx')
    session = ort.InferenceSession(f'{MODELES}/{nom}.onnx', providers=['CPUExecutionProvider'])
    pred_onnx = session.run(None, {session.get_inputs()[0].name: X_test.values})[0]
    lignes.append({'modèle': nom,
                   'pickle (Ko)': round(os.path.getsize(f'{MODELES}/{nom}.pkl') / 1e3),
                   'onnx (Ko)': round(os.path.getsize(f'{MODELES}/{nom}.onnx') / 1e3),
                   'mêmes prédictions': f'{(pred_onnx == m.predict(X_test)).mean():.2%}',
                   'opérateurs ONNX': ', '.join(sorted({n.op_type for n in onx.graph.node}))})
pd.DataFrame(lignes)

,modèle,pickle (Ko),onnx (Ko),mêmes prédictions,opérateurs ONNX
0,arbre,11,5,100.00%,TreeEnsembleClassifier
1,foret,1411,664,100.00%,TreeEnsembleClassifier
2,knn,4960,4850,100.00%,"ArgMax, ArrayFeatureExtractor, Cast, Concat, D..."


**👀 Ce qu'on doit voir :**
- **100 %** de prédictions identiques : la conversion ne change pas le modèle ;
- l'arbre et la forêt deviennent un seul opérateur `TreeEnsembleClassifier`, et le fichier est plus petit ;
- le KNN devient des opérateurs de calcul (`TopK`, `Sqrt`, …) et le fichier garde à peu près la même taille : **il contient toujours toute la base d'entraînement**. Un format sûr protège contre l'exécution de code, pas contre la fuite de données.

## Étape 6 : safetensors, le format des réseaux de neurones
**Ce qu'on fait :** pour un réseau de neurones, les poids sont de simples tableaux (tenseurs). safetensors = un en-tête JSON + les octets bruts.
On télécharge le `model.safetensors` du petit BERT de l'exercice 1 et on lit son en-tête, toujours dans un processus isolé.

In [10]:
from helper import telecharger_hf, analyser_isole
fichiers_bert = telecharger_hf([('hf-internal-testing/tiny-random-bert', 'model.safetensors', 'Petit BERT (safetensors)'),
                                ('hf-internal-testing/tiny-random-bert', 'pytorch_model.bin', 'Petit BERT (PyTorch)')])
st = analyser_isole('safetensors', fichiers_bert[0]['chemin'])
pt = analyser_isole('imports', fichiers_bert[1]['chemin'])
print(f"safetensors : {st['nb_tenseurs']} tenseurs, en-tête de {st['taille_entete']} octets, métadonnées = {st['metadata']}")
print(f"pytorch_model.bin : imports demandés = {pt['imports']}")
pd.DataFrame(st['tenseurs']).T.head(5)

safetensors : 100 tenseurs, en-tête de 10752 octets, métadonnées = {'format': 'pt', 'cls.predictions.decoder.bias': 'cls.predictions.bias'}
pytorch_model.bin : imports demandés = ['collections.OrderedDict', 'torch.FloatStorage', 'torch.LongStorage', 'torch._utils._rebuild_tensor_v2']


,dtype,shape
bert.embeddings.position_ids,I64,"[1, 512]"
bert.embeddings.LayerNorm.bias,F32,[32]
bert.embeddings.LayerNorm.weight,F32,[32]
bert.embeddings.position_embeddings.weight,F32,"[512, 32]"
bert.embeddings.token_type_embeddings.weight,F32,"[16, 32]"


**👀 Ce qu'on doit voir :** le fichier safetensors ne déclare **aucun import** : seulement des noms de tenseurs, des types (`F32`) et des formes.
Le même modèle au format PyTorch a besoin d'importer des fonctions `torch` pour se reconstruire.

**❓ Question pour la classe :** pourquoi ne pas utiliser safetensors pour la forêt aléatoire ? *Réponse : une forêt n'est pas qu'un ensemble de tableaux ; il faudrait écrire soi-même le code qui la reconstruit. ONNX standardise ce code sous forme d'opérateurs.*

## Ce qu'il faut retenir
- Le fichier d'un modèle, c'est **ce que l'attaquant peut modifier** : des seuils (arbre, forêt), ou carrément la base d'entraînement (KNN). La précision globale ne révèle pas une modification ciblée.
- **Empreinte SHA-256** = le fichier n'a pas changé. **Signature** = il vient bien de nous. Il faut les deux, et vérifier **avant** de charger.
- **Formats sans code** : ONNX pour scikit-learn, safetensors pour les réseaux de neurones. Ils suppriment l'exécution de code au chargement, pas la fuite de données.

**Exercice suivant :** le modèle n'est qu'un maillon. On inventorie tout l'environnement (ML-BOM) et on cherche ses vulnérabilités connues.